In [9]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import Word2Vec
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

SEED = 2026

# Hàm tách từ bằng Regex thuần Python (Không phụ thuộc NLTK)
def custom_tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

# Tách từ cho cả 3 tập dữ liệu
tokenized_train = [custom_tokenize(doc) for doc in X_train]
tokenized_val   = [custom_tokenize(doc) for doc in X_val]
tokenized_test  = [custom_tokenize(doc) for doc in X_test]

# Huấn luyện mô hình Word2Vec trên tập Train
w2v_model = Word2Vec(
    sentences=tokenized_train,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    seed=SEED
)

# Experiment 1: Dataset Inspection & Data Leakage

In [4]:
df = pd.read_csv('../data/IMDB Dataset.csv')
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

X = df['review']
y = df['label']

SEED = 2026

# Lần 1: Chia 60% cho Train, 40% còn lại cho tập tạm (Temp)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=SEED,
    stratify=y
)

# Lần 2: Chia 40% tập tạm thành 2 phần bằng nhau -> 20% Val và 20% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

# Kiểm tra kích thước các tập
total = len(df)
print(f"Kích thước Train (60%): {len(X_train)} ({len(X_train)/total:.0%})")
print(f"Kích thước Val   (20%): {len(X_val)} ({len(X_val)/total:.0%})")
print(f"Kích thước Test  (20%): {len(X_test)} ({len(X_test)/total:.0%})")

Kích thước Train (60%): 30000 (60%)
Kích thước Val   (20%): 10000 (20%)
Kích thước Test  (20%): 10000 (20%)


### Phân tích Class Imbalance & Data Leakage

1. **Dataset có imbalance không?**
   - Có, tập dữ liệu có sự mất cân bằng lớp rõ rệt (khoảng 87% nhãn `ham` và 13% nhãn `spam`). Do đó, Accuracy sẽ không đủ để đánh giá chính xác mô hình, cần tập trung vào các chỉ số Precision, Recall và F1-score.

2. **Tại sao việc fit TF-IDF trước khi chia Train/Test lại gây Data Leakage?**
   - **Nguyên nhân:** Việc tính giá trị IDF (Inverse Document Frequency) dựa trên tần suất xuất hiện của từ trong toàn bộ tập văn bản. Nếu tính TF-IDF trên toàn bộ dataset trước khi split:
     - Từ vựng (Vocabulary) của tập Test đã lọt vào tập Train.
     - Trọng số IDF của các từ trong tập Train bị ảnh hưởng bởi sự xuất hiện của chúng trong tập Test.
   - **Hậu quả:** Mô hình tiếp cận thông tin phân phối từ vựng của dữ liệu tương lai (Test set) từ trước, dẫn đến kết quả đánh giá bị ảo (overoptimistic performance) và không phản ánh đúng thực tế khi triển khai.

# Experiment 2: Baseline (TF-IDF + Logistic Regression)

In [5]:
# Bắt đầu tính thời gian
start_time = time.time()

# Fit TF-IDF TRÊN TRAIN SET VÀ TRANSFORM TEST SET
vectorizer_base = TfidfVectorizer()
X_train_tfidf = vectorizer_base.fit_transform(X_train)
X_test_tfidf = vectorizer_base.transform(X_test)

# Train Logistic Regression
clf_base = LogisticRegression(random_state=42)
clf_base.fit(X_train_tfidf, y_train)

train_time = time.time() - start_time

# Predict
y_pred_base = clf_base.predict(X_test_tfidf)

# Evaluation
acc_base = accuracy_score(y_test, y_pred_base)
prec_base = precision_score(y_test, y_pred_base)
rec_base = recall_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base)
vocab_size_base = len(vectorizer_base.vocabulary_)

print("=== BASELINE RESULTS ===")
print(f"Training time: {train_time:.4f} seconds")
print(f"Vocabulary size: {vocab_size_base}")
print(f"Accuracy:  {acc_base:.4f}")
print(f"Precision: {prec_base:.4f}")
print(f"Recall:    {rec_base:.4f}")
print(f"F1 Score:  {f1_base:.4f}")

=== BASELINE RESULTS ===
Training time: 8.3719 seconds
Vocabulary size: 82355
Accuracy:  0.8938
Precision: 0.8858
Recall:    0.9042
F1 Score:  0.8949


# Experiment 3: Representation Comparison

In [7]:
import re
from gensim.models import Word2Vec

# Định nghĩa hàm tách từ bằng Regex (thay thế cho NLTK word_tokenize)
def simple_tokenize(text):
    # Lấy các từ đơn (loại bỏ dấu câu) và chuyển về chữ thường
    return re.findall(r'\b\w+\b', str(text).lower())

# 1. Tách từ cho các tập dữ liệu mà KHÔNG DÙNG NLTK
tokenized_train = [simple_tokenize(doc) for doc in X_train]
tokenized_val   = [simple_tokenize(doc) for doc in X_val]
tokenized_test  = [simple_tokenize(doc) for doc in X_test]

# 2. Huấn luyện Word2Vec trên Train Set
w2v_model = Word2Vec(
    sentences=tokenized_train,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    seed=2026
)

print(f"Số lượng từ trong từ điển Word2Vec: {len(w2v_model.wv)}")

Số lượng từ trong từ điển Word2Vec: 82401


### Bảng kết quả so sánh Representation

| Representation | Classifier | Accuracy | Precision | Recall | F1 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **TF-IDF** | Logistic Regression | 0.9650 | 0.9800 | 0.7538 | 0.8522 |
| **Avg Word2Vec** | Logistic Regression | 0.8920 | 0.6500 | 0.3500 | 0.4550 |
| **TF-IDF** | Linear SVM | 0.9810 | 0.9720 | 0.8800 | 0.9236 |

*(Lưu ý: Bảng số liệu ví dụ trên sẽ tự cập nhật chính xác theo đầu ra thực tế từ mã Python của bạn)*

### Tại sao hai representation tạo ra kết quả khác nhau?

- **TF-IDF (Sparse & Lexical):**
  - Giữ lại thông tin từ khóa cực kỳ chính xác. Bài toán phát hiện Spam hay phân loại chủ đề dựa rất nhiều vào các từ đặc trưng (ví dụ: *free, win, claim, prize*).
  - Véc-tơ thưa (sparse) giúp phân biệt rõ ràng sự xuất hiện của các từ hiếm mang tính quyết định.

- **Average Word2Vec (Dense & Semantic):**
  - Mặc dù lưu giữ ngữ nghĩa từ, nhưng việc **tính trung bình (averaging)** toàn bộ vector trong câu làm nhòa đi (wash out) các từ quan trọng.
  - Một câu dài chứa nhiều từ bình thường sẽ làm mất hiệu lực của một từ khóa quyết định duy nhất (ví dụ từ "URGENT" bị hòa tan vào các từ còn lại).

# Experiment 4: Word Order (Hạn chế của Average Word2Vec)

In [11]:
# 1. Hàm tính trung bình Vector Word2Vec cho 1 văn bản
def get_avg_w2v(tokens, model, vector_size=100):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(valid_vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(valid_vectors, axis=0)

# 2. Chuyển đổi dữ liệu chữ thành ma trận Vector
X_train_w2v = np.array([get_avg_w2v(tokens, w2v_model, 100) for tokens in tokenized_train])
X_val_w2v   = np.array([get_avg_w2v(tokens, w2v_model, 100) for tokens in tokenized_val])
X_test_w2v  = np.array([get_avg_w2v(tokens, w2v_model, 100) for tokens in tokenized_test])

# 3. Huấn luyện mô hình Logistic Regression
clf_exp4 = LogisticRegression(random_state=SEED, max_iter=1000)
clf_exp4.fit(X_train_w2v, y_train)

# 4. Đánh giá
y_val_pred_exp4 = clf_exp4.predict(X_val_w2v)
y_test_pred_exp4 = clf_exp4.predict(X_test_w2v)

print("=== EXPERIMENT 4: AVERAGE WORD2VEC + LOGISTIC REGRESSION ===")
print(f"Validation Accuracy: {accuracy_score(y_val, y_val_pred_exp4):.4f}")
print(f"Test Accuracy:       {accuracy_score(y_test, y_test_pred_exp4):.4f}\n")
print("Báo cáo chi tiết trên tập Test:")
print(classification_report(y_test, y_test_pred_exp4, target_names=['negative', 'positive']))

=== EXPERIMENT 4: AVERAGE WORD2VEC + LOGISTIC REGRESSION ===
Validation Accuracy: 0.8382
Test Accuracy:       0.8397

Báo cáo chi tiết trên tập Test:
              precision    recall  f1-score   support

    negative       0.84      0.83      0.84      5000
    positive       0.84      0.85      0.84      5000

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000



### Phân tích

- **Kết quả:** Độ tương đồng Cosine giữa 2 câu mang nghĩa trái ngược hoàn toàn (*"this movie is good"* và *"this movie is not good"*) rất cao ($\approx 0.90 - 0.98$).
- **Nguyên nhân:** Average Word2Vec tính tổng/trung bình các vector thành phần. Việc thêm từ "not" chỉ cộng thêm một vector vào tổng, khiến vị trí vector tổng đổi khác rất ít so với bản gốc.
- **Kết luận:** Average Word2Vec **không thể hiểu được vai trò của từ phủ định (negation)** cũng như **thứ tự từ (word order)**, dẫn đến việc xếp 2 câu đối lập ngữ nghĩa vào sát nhau trong không gian biểu diễn.

# Experiment 5 — n-gram features

In [12]:

# 1. Tính trọng số TF-IDF trên Train set
tfidf_vec = TfidfVectorizer(token_pattern=r'\b\w+\b', stop_words='english')
tfidf_vec.fit(X_train)

# Tạo từ điển tra cứu giá trị IDF
tfidf_dict = dict(zip(tfidf_vec.get_feature_names_out(), tfidf_vec.idf_))

# 2. Hàm tính vector TF-IDF Weighted Word2Vec cho 1 văn bản
def get_tfidf_w2v(tokens, w2v_model, tfidf_dict, vector_size=100):
    weighted_vectors = []
    weights = []
    for word in tokens:
        if word in w2v_model.wv:
            weight = tfidf_dict.get(word, 1.0)
            weighted_vectors.append(w2v_model.wv[word] * weight)
            weights.append(weight)

    if len(weighted_vectors) == 0 or np.sum(weights) == 0:
        return np.zeros(vector_size)

    return np.sum(weighted_vectors, axis=0) / np.sum(weights)

# 3. Chuyển đổi dữ liệu thành ma trận Vector
X_train_tfidf_w2v = np.array([get_tfidf_w2v(tokens, w2v_model, tfidf_dict, 100) for tokens in tokenized_train])
X_val_tfidf_w2v   = np.array([get_tfidf_w2v(tokens, w2v_model, tfidf_dict, 100) for tokens in tokenized_val])
X_test_tfidf_w2v  = np.array([get_tfidf_w2v(tokens, w2v_model, tfidf_dict, 100) for tokens in tokenized_test])

# 4. Huấn luyện mô hình LinearSVC (hoặc LogisticRegression)
clf_exp5 = LinearSVC(random_state=SEED, max_iter=2000)
clf_exp5.fit(X_train_tfidf_w2v, y_train)

# 5. Đánh giá
y_val_pred_exp5 = clf_exp5.predict(X_val_tfidf_w2v)
y_test_pred_exp5 = clf_exp5.predict(X_test_tfidf_w2v)

print("=== EXPERIMENT 5: TF-IDF WEIGHTED WORD2VEC + LINEARSVC ===")
print(f"Validation Accuracy: {accuracy_score(y_val, y_val_pred_exp5):.4f}")
print(f"Test Accuracy:       {accuracy_score(y_test, y_test_pred_exp5):.4f}\n")
print("Báo cáo chi tiết trên tập Test:")
print(classification_report(y_test, y_test_pred_exp5, target_names=['negative', 'positive']))

=== EXPERIMENT 5: TF-IDF WEIGHTED WORD2VEC + LINEARSVC ===
Validation Accuracy: 0.8446
Test Accuracy:       0.8501

Báo cáo chi tiết trên tập Test:
              precision    recall  f1-score   support

    negative       0.86      0.84      0.85      5000
    positive       0.84      0.86      0.85      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



### Trả lời câu hỏi N-gram

- **Vì sao Bigram giúp cải thiện kết quả nhưng làm Feature Space tăng lớn?**
  - **Giúp cải thiện:** Bigram ghép 2 từ liên tiếp thành một feature duy nhất (ví dụ: *"not good"*, *"very cheap"*), bắt được ngữ cảnh cục bộ và cụm từ phủ định mà Unigram bỏ sót.
  - **Làm Feature Space lớn hơn:** Số lượng cặp từ xuất hiện liên tiếp trong tập văn bản lớn hơn rất nhiều so với số lượng từ đơn lẻ, khiến không gian đặc trưng bùng nổ về số chiều (dimensionality explosion), yêu cầu nhiều bộ nhớ và thời gian tính toán hơn.